In [5]:
import os
import httpx
import time
from llama_index.core import (
    VectorStoreIndex,
    SimpleKeywordTableIndex,  # 替換：改引入 SimpleKeywordTableIndex
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
    Settings,
    PromptTemplate
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.node_parser import TokenTextSplitter
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("llama_index").setLevel(logging.WARNING)

# 1. 系統硬體環境初始化確認
def check_ollama_service():
    url = "http://localhost:11434"
    try:
        response = httpx.get(url)
        if response.status_code == 200:
            print("✅ Ollama 服務器 http://localhost:11434 可達。")
            return True
    except httpx.RequestError:
        print("❌ 無法連接到 Ollama 服務器，請檢查 Ollama 是否已啟動。")
        return False

# 2. 定義 RAG 核心類別
class KeywordTableRAGSystem:
    def __init__(self, input_file, idx_path, model_name="gpt-oss:20b", embed_model_name="bge-m3"):
        self.input_file = input_file
        self.idx_path = idx_path
        self.model_name = model_name
        self.embed_model_name = embed_model_name
        self.index = None
        self.query_engine = None
        
        # 初始化 LlamaIndex 的全域 Settings
        self._setup_settings()
        
    def _setup_settings(self):
        # 配置 LLM
        Settings.llm = Ollama(
            model=self.model_name,
            request_timeout=300.0,
            baseUrl="http://localhost:11434"
        )
        
        # 配置 Embedding Model
        Settings.embed_model = OllamaEmbedding(
            model_name=self.embed_model_name,
            base_url="http://localhost:11434",
        )
        
        # 配置切分參數（已修改為 4096 / 0）
        Settings.node_parser = TokenTextSplitter(
            chunk_size=4096,
            chunk_overlap=0
        )

    def initialize_index(self):
        """建立或讀取現有的 SimpleKeywordTableIndex"""
        if os.path.exists(self.idx_path):
            print(f"✅ 成功載入現有索引於 '{self.idx_path}'。")
            storage_context = StorageContext.from_defaults(persist_dir=self.idx_path)
            self.index = load_index_from_storage(storage_context)
        else:
            print(f"🔍 找不到現有索引，開始讀取 '{self.input_file}' 並建立 SimpleKeywordTableIndex...")
            if not os.path.exists(self.input_file):
                raise FileNotFoundError(f"❌ 錯誤: 找不到輸入檔案 {self.input_file}")
                
            # 讀取單一文件
            documents = SimpleDirectoryReader(input_files=[self.input_file]).load_data()
            
            # 建立 SimpleKeywordTableIndex 索引
            self.index = SimpleKeywordTableIndex.from_documents(documents)
            
            # 持久化儲存索引
            self.index.storage_context.persist(persist_dir=self.idx_path)
            print(f"💾 索引建立完成並已成功儲存至 '{self.idx_path}'。")

    def setup_query_engine(self):
        """設定查詢引擎與指定的繁體中文 qa_template"""
        # 定義指定的 QA Template
        qa_prompt_tmpl_str = (
            "請根據提供的參考資料回答用戶問題。\n\n"
            "【參考資料】\n"
            "---------------------\n"
            "{context_str}\n"
            "---------------------\n"
            "問題：{query_str}\n\n"
            "【指令規範】\n"
            "1. **全面提取**：若資料包含答案，請以繁體中文清楚、完整地列出不得遺漏。\n"
            "2. **忠於原文**：保持客觀，絕對不能因總結而遺漏任何資訊，優先使用資料內的用語。\n"
            "3. **誠實原則**：如果資料中確實完全沒有提到相關資訊，請禮貌地回答「根據目前的資料庫，無法提供相關資訊」。\n"
            "4. **完整性**：忽略資料中不相關的雜訊（如：分頁標記、無意義的編號）。\n"
            "5. **後驗檢查**：在輸出答案前，請再次確認是否遺漏了【參考資料】中任何符合問題條件的細節。"
        )
        qa_template = PromptTemplate(qa_prompt_tmpl_str)

        # 建立查詢引擎並套用樣板
        self.query_engine = self.index.as_query_engine(
            text_qa_template=qa_template
        )

    def rag_query(self, query_str):
        """執行 RAG 查詢並回傳文字答案"""
        if not self.query_engine:
            raise ValueError("❌ 查詢引擎尚未初始化，請先呼叫 setup_query_engine()")
        
        response = self.query_engine.query(query_str)
        return str(response)

    def show_index_info(self):
        """列印 SimpleKeywordTableIndex 結構資訊"""
        print("\n📋 SimpleKeywordTableIndex 架構:")
        print(" ====================")
        # 取得關鍵字表格對應的 Node 數量與內容
        table = self.index.index_struct.table
        print(f"共有 {len(table)} 個獨特關鍵字索引。")
        print(" 列印部分關鍵字與節點對應摘要:")
        print("--------------------------------------------------")
        # 列出前 5 個關鍵字與對應的 Node ID 作為架構展示
        for idx, (keyword, node_ids) in enumerate(list(table.items())[:]):
            print(f"關鍵字: {keyword} -> 連結的 Node 數量: {len(node_ids)}")
        print("--------------------------------------------------\n")


# 3. 主程式測試進入點
def main(input_file, idx_path, model_name, embed_model_name):
    # 檢查 Ollama 連線狀況
    if not check_ollama_service():
        return
        
    print(f"🤖 使用 LlamaIndex SimpleKeywordTableIndex 的 {model_name} RAG系統測試")
    print(" ==================================================")
    
    # 建立系統實例
    rag = KeywordTableRAGSystem(
        input_file=input_file,
        idx_path=idx_path,
        model_name=model_name,
        embed_model_name=embed_model_name
    )
    
    # 初始化索引與查詢引擎
    rag.initialize_index()
    rag.setup_query_engine()
    
    # 展示索引內部的關鍵字對應結構
    rag.show_index_info()
    
    # 進入無窮迴圈進行互動式測試
    while True:
        query = input("❓ 請輸入測試問題 (輸入 quit 結束): ").strip()
        
        if query.lower() == "quit":
            print("\n✅ 結束測試。")
            break
            
        if not query:
            continue
            
        start_time = time.time()
        answer = rag.rag_query(query)
        elapsed_seconds = time.time() - start_time
        
        print(f"🤖 回答: {answer}\n", "-" * 50)
        print(f"⏱️ 查詢花費時間: {elapsed_seconds:.4f} 秒\n", "-" * 50)


if __name__ == "__main__":
    # 配置輸入檔案與儲存資料夾路徑
    main(
        input_file="114_碩士班招生簡章_分頁.txt", 
        idx_path="app6_keyword_storage",  # 儲存目錄名稱
        model_name="gpt-oss:20b",
        embed_model_name="bge-m3"
    )

✅ Ollama 服務器 http://localhost:11434 可達。
🤖 使用 LlamaIndex SimpleKeywordTableIndex 的 gpt-oss:20b RAG系統測試
✅ 成功載入現有索引於 'app6_keyword_storage'。

📋 SimpleKeywordTableIndex 架構:
共有 53 個獨特關鍵字索引。
 列印部分關鍵字與節點對應摘要:
--------------------------------------------------
關鍵字: 頁 -> 連結的 Node 數量: 4
關鍵字: 月 -> 連結的 Node 數量: 7
關鍵字: 附表 -> 連結的 Node 數量: 1
關鍵字: 年 -> 連結的 Node 數量: 10
關鍵字: 日 -> 連結的 Node 數量: 2
關鍵字: 1 -> 連結的 Node 數量: 8
關鍵字: 113 -> 連結的 Node 數量: 1
關鍵字: 3 -> 連結的 Node 數量: 5
關鍵字: 114 -> 連結的 Node 數量: 8
關鍵字: 2 -> 連結的 Node 數量: 7
關鍵字: 一 -> 連結的 Node 數量: 8
關鍵字: 力 -> 連結的 Node 數量: 1
關鍵字: 二 -> 連結的 Node 數量: 8
關鍵字: 繳交 -> 連結的 Node 數量: 1
關鍵字: 書面審查 -> 連結的 Node 數量: 2
關鍵字: 報考資格 -> 連結的 Node 數量: 1
關鍵字: 三 -> 連結的 Node 數量: 5
關鍵字: 17 -> 連結的 Node 數量: 1
關鍵字: 書面審查一 -> 連結的 Node 數量: 1
關鍵字: shu -> 連結的 Node 數量: 4
關鍵字: 方式 -> 連結的 Node 數量: 2
關鍵字: tw -> 連結的 Node 數量: 4
關鍵字: edu -> 連結的 Node 數量: 4
關鍵字: 書面審查二 -> 連結的 Node 數量: 1
關鍵字: 4 -> 連結的 Node 數量: 1
關鍵字: https -> 連結的 Node 數量: 1
關鍵字: 第 -> 連結的 Node 數量: 7
關鍵字: 持國外或香港 -> 連結的 Node 數量: 1
關鍵字: 條 -> 

❓ 請輸入測試問題 (輸入 quit 結束):  資管系招生名額多少?


🤖 回答: Empty Response
 --------------------------------------------------
⏱️ 查詢花費時間: 15.1092 秒
 --------------------------------------------------


❓ 請輸入測試問題 (輸入 quit 結束):  quit



✅ 結束測試。
